In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [13]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score


#loading file path
hp_train_file_path = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
hp_test_file_path = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'

#reading file path
hp_train_data = pd.read_csv(hp_train_file_path)

#creating features(x) and target(y)
y = hp_train_data.SalePrice
X = hp_train_data.drop(['SalePrice','Id'],axis = 1)

#splitting the data for validation
X_train, X_valid, y_train , y_valid = train_test_split(X,y,random_state = 0)

#using a imputer for missing numerical values
hp_imputer_num = SimpleImputer(strategy = 'median')
hp_imputer_cat = SimpleImputer(strategy = 'constant')

#splitting the data into numerical and categorical values

hp_num_cols = [c for c in X_train.columns if X_train[c].dtype in ['int64','float64']]
hp_cat_cols = [c for c in X_train.columns if X_train[c].dtype =='object']

#imputting data to handle missing values for numerical first

imputed_X_num_train = pd.DataFrame(hp_imputer_num.fit_transform(X_train[hp_num_cols]))
imputed_X_num_valid = pd.DataFrame(hp_imputer_num.transform(X_valid[hp_num_cols]))

#adding names to the columns. its not x_train.columns as it contains cat columns to so
#its important we add it to num colos only
imputed_X_num_train.columns = hp_num_cols
imputed_X_num_valid.columns = hp_num_cols

#giving making we are making the original index and the imputed index the same as test train
#split ended up shuffling it

imputed_X_num_train.index = X_train.index
imputed_X_num_valid.index = X_valid.index

#imputing for categorical values
imputed_X_cat_train = pd.DataFrame(hp_imputer_cat.fit_transform(X_train[hp_cat_cols]))
imputed_X_cat_valid = pd.DataFrame(hp_imputer_cat.transform(X_valid[hp_cat_cols]))

#adding names
imputed_X_cat_train.columns = hp_cat_cols
imputed_X_cat_valid.columns = hp_cat_cols

#adding index
imputed_X_cat_train.index = X_train.index
imputed_X_cat_valid.index = X_valid.index 

print(imputed_X_num_train.isnull().sum().sum(), imputed_X_cat_train.isnull().sum().sum())   # 0 0
print(imputed_X_cat_valid.index[:5], X_valid.index[:5])

#hot encoding for categorical data

hp_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output = False)

#gets the names out 
encoded_train = pd.DataFrame(hp_encoder.fit_transform(imputed_X_cat_train),
                            columns = hp_encoder.get_feature_names_out(hp_cat_cols),
                            index=X_train.index)
encoded_valid = pd.DataFrame(hp_encoder.transform(imputed_X_cat_valid),
                             columns=hp_encoder.get_feature_names_out(hp_cat_cols),
                             index=X_valid.index)

final_X_train = pd.concat([imputed_X_num_train, encoded_train],axis = 1)
final_X_valid = pd.concat([imputed_X_num_valid, encoded_valid],axis = 1)

hp_model = RandomForestRegressor(random_state=1)
hp_model.fit(final_X_train,y_train)
preds = hp_model.predict(final_X_valid)
print(mean_absolute_error(y_valid,preds))

def cv_mae(cat_imputer):
    preprocessing =ColumnTransformer([
        ('num', SimpleImputer(strategy='median'),hp_num_cols),
        ('cat', Pipeline([('imp',cat_imputer),
                         ('oh',OneHotEncoder(handle_unknown='ignore'))]),hp_cat_cols)
        
    ])
    pipe = Pipeline([('preprocessing',preprocessing),('rf',RandomForestRegressor(random_state=1))])
    return -1*cross_val_score(pipe,X,y,cv=5,
                             scoring ='neg_mean_absolute_error').mean()

print(cv_mae(SimpleImputer(strategy='most_frequent')))
print(cv_mae(SimpleImputer(strategy='constant', fill_value='None')))



0 0
Index([529, 491, 459, 279, 655], dtype='int64') Index([529, 491, 459, 279, 655], dtype='int64')
17052.632493150686
17672.27747945206
17690.512349315068
